In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim
import numpy as np
import pandas as pd
from enum import Enum
import math
import random
from collections import defaultdict

# -----------------------
# Ground Truth Generator
# -----------------------

class HamburgCrisisLevel(Enum):
    NORMAL = 0
    STRESSED = 1
    CRISIS = 2
    BREAKDOWN = 3
    STAFF_EXODUS = 4

class HamburgGroundTruthGenerator:
    def __init__(self):
        self.total_ambulances = 80
        self.crisis_thresholds = {
            'normal_max': 10,
            'stressed_max': 15,
            'crisis_max': 20,
            'breakdown_min': 20
        }

    def determine_crisis_level(self, crew_calls_today, system_calls_today, hour, available_crews):
        calls_per_crew_avg = system_calls_today / self.total_ambulances
        availability_ratio = available_crews / self.total_ambulances
        crew_stress_factor = crew_calls_today / 24.0

        if 20 <= hour or hour <= 6:
            time_multiplier = 1.3
        elif 14 <= hour <= 18:
            time_multiplier = 1.2
        else:
            time_multiplier = 1.0

        if availability_ratio < 0.3:
            availability_multiplier = 1.5
        elif availability_ratio < 0.5:
            availability_multiplier = 1.2
        else:
            availability_multiplier = 1.0

        effective_calls = crew_calls_today * time_multiplier * availability_multiplier

        if effective_calls >= 25 or crew_calls_today >= 22:
            return HamburgCrisisLevel.STAFF_EXODUS
        elif effective_calls >= 20 or crew_calls_today >= 20:
            return HamburgCrisisLevel.BREAKDOWN
        elif effective_calls >= 15 or crew_calls_today >= 15:
            return HamburgCrisisLevel.CRISIS
        elif effective_calls >= 10 or crew_calls_today >= 10:
            return HamburgCrisisLevel.STRESSED
        else:
            return HamburgCrisisLevel.NORMAL

    def calculate_burnout_risk(self, crew_calls_today, crew_hours_worked, crisis_level, consecutive_high_days=1):
        base_risk = 0.1
        if crew_calls_today >= 22:
            call_risk = 0.7
        elif crew_calls_today >= 20:
            call_risk = 0.5
        elif crew_calls_today >= 15:
            call_risk = 0.3
        elif crew_calls_today >= 10:
            call_risk = 0.15
        else:
            call_risk = 0.0

        if crew_hours_worked >= 12:
            hour_risk = 0.2
        elif crew_hours_worked >= 10:
            hour_risk = 0.1
        else:
            hour_risk = 0.0

        crisis_multipliers = {
            HamburgCrisisLevel.NORMAL: 1.0,
            HamburgCrisisLevel.STRESSED: 1.2,
            HamburgCrisisLevel.CRISIS: 1.5,
            HamburgCrisisLevel.BREAKDOWN: 2.0,
            HamburgCrisisLevel.STAFF_EXODUS: 3.0
        }

        consecutive_risk = min(0.3, consecutive_high_days * 0.05)
        total_risk = (base_risk + call_risk + hour_risk + consecutive_risk) * crisis_multipliers[crisis_level]
        return min(1.0, total_risk)

    def calculate_staff_retention_probability(self, burnout_risk, crisis_level, peak_calls_last_week, system_support_quality=0.5):
        base_retention = 0.8
        burnout_penalty = burnout_risk * 0.6
        crisis_penalties = {
            HamburgCrisisLevel.NORMAL: 0.0,
            HamburgCrisisLevel.STRESSED: 0.1,
            HamburgCrisisLevel.CRISIS: 0.25,
            HamburgCrisisLevel.BREAKDOWN: 0.4,
            HamburgCrisisLevel.STAFF_EXODUS: 0.6
        }
        if peak_calls_last_week >= 25:
            peak_penalty = 0.3
        elif peak_calls_last_week >= 22:
            peak_penalty = 0.2
        elif peak_calls_last_week >= 20:
            peak_penalty = 0.1
        else:
            peak_penalty = 0.0

        support_bonus = (system_support_quality - 0.5) * 0.2
        retention_prob = (base_retention - burnout_penalty - crisis_penalties[crisis_level] - peak_penalty + support_bonus)
        return max(0.1, min(1.0, retention_prob))

    def calculate_response_delay(self, crisis_level, available_crews, calls_waiting, time_of_day, weather_factor=1.0):
        baseline_response = 8.0
        crisis_multipliers = {
            HamburgCrisisLevel.NORMAL: 1.0,
            HamburgCrisisLevel.STRESSED: 1.3,
            HamburgCrisisLevel.CRISIS: 1.8,
            HamburgCrisisLevel.BREAKDOWN: 2.5,
            HamburgCrisisLevel.STAFF_EXODUS: 3.5
        }
        availability_ratio = available_crews / 80.0
        if availability_ratio < 0.2:
            availability_factor = 3.0
        elif availability_ratio < 0.4:
            availability_factor = 2.0
        elif availability_ratio < 0.6:
            availability_factor = 1.5
        else:
            availability_factor = 1.0
        queue_factor = 1.0 + (calls_waiting * 0.2)

        if 2 <= time_of_day <= 6:
            time_factor = 1.4
        elif 14 <= time_of_day <= 18:
            time_factor = 1.2
        elif 20 <= time_of_day or time_of_day <= 2:
            time_factor = 1.3
        else:
            time_factor = 1.0

        total_delay = (baseline_response * crisis_multipliers[crisis_level] *
                       availability_factor * queue_factor * time_factor * weather_factor)
        return min(60.0, total_delay)

    def calculate_safety_degradation(self, crew_calls_today, crew_fatigue_level, crisis_level, response_delay):
        base_handoff_quality = 0.95
        base_documentation = 0.90
        base_clinical_care = 0.92

        if crew_calls_today >= 22:
            call_degradation = 0.4
        elif crew_calls_today >= 20:
            call_degradation = 0.3
        elif crew_calls_today >= 15:
            call_degradation = 0.2
        elif crew_calls_today >= 10:
            call_degradation = 0.1
        else:
            call_degradation = 0.0

        fatigue_degradation = crew_fatigue_level * 0.15
        crisis_degradation = {
            HamburgCrisisLevel.NORMAL: 0.0,
            HamburgCrisisLevel.STRESSED: 0.05,
            HamburgCrisisLevel.CRISIS: 0.15,
            HamburgCrisisLevel.BREAKDOWN: 0.25,
            HamburgCrisisLevel.STAFF_EXODUS: 0.4
        }[crisis_level]

        if response_delay > 20:
            delay_degradation = 0.2
        elif response_delay > 15:
            delay_degradation = 0.1
        elif response_delay > 10:
            delay_degradation = 0.05
        else:
            delay_degradation = 0.0

        total_degradation = call_degradation + fatigue_degradation + crisis_degradation + delay_degradation

        return {
            'handoff_quality': max(0.3, base_handoff_quality - total_degradation),
            'documentation_completeness': max(0.4, base_documentation - total_degradation * 0.8),
            'clinical_care_quality': max(0.5, base_clinical_care - total_degradation * 0.6),
            'adverse_event_risk': min(0.8, total_degradation * 0.7),
            'patient_satisfaction': max(0.2, 0.85 - total_degradation * 1.2)
        }

    def _estimate_time_to_breakdown(self, current_crisis, available_crews):
        if current_crisis == HamburgCrisisLevel.BREAKDOWN or current_crisis == HamburgCrisisLevel.STAFF_EXODUS:
            return 0.0
        elif current_crisis == HamburgCrisisLevel.CRISIS:
            return max(1.0, 6.0 - (available_crews / 20.0))
        elif current_crisis == HamburgCrisisLevel.STRESSED:
            return max(2.0, 12.0 - (available_crews / 10.0))
        else:
            return 24.0

    def _calculate_escalation_risk(self, current_crisis, crew_calls):
        base_esca_
